# Ch.3 — Feature Scaling, Importance & Multicollinearity

**Goal:** SmartVal AI has 8 features and a $55k MAE from Ch.2. Before adding polynomial complexity (Ch.4) or regularisation (Ch.5), we need to answer:
- Why must we scale features before comparing their importance?
- Which features carry the most signal toward house value?
- Which features measure the same thing (and therefore compete for the same weights)?
- What does each diagnostic method reveal that the others miss?

By the end of this notebook you will have a **three-view feature dashboard** and specific action items for Ch.4 and Ch.5.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import the required libraries: matplotlib.gridspec, matplotlib.pyplot, numpy, pandas, seaborn, sklearn.datasets
# 2. Set random seed (np.random.seed) and configure the plot style
#
# Hint:
#   import numpy as np
#   import pandas as pd
#   import matplotlib.pyplot as plt

## 0 · Feature Scaling

Before any importance ranking is possible, all features must be on a **common scale**. Raw weights from an unstandardized model are not comparable — a weight of 200 for `Population` (range 3–35,682) looks enormous next to a weight of 0.4 for `MedInc` (range 0.5–15), but that gap reflects the input scale, not the feature's importance.

Think of this like comparing heights measured in millimeters vs kilometers — a 1.8m person becomes 1,800 mm but 0.0018 km. The number changes dramatically, but the person stays exactly the same height. Similarly, a feature's raw weight depends entirely on what units you happened to choose for measurement. StandardScaler fixes this by converting every feature to the same unit: **"one typical swing in that feature across the dataset."** After standardization, a coefficient of 0.8 on income and 0.1 on population means income has genuinely 8× the effect per standard deviation of change — a fair comparison.

**Standardization (Z-score):** $x_j^{\text{std}} = \dfrac{x_j - \mu_j}{\sigma_j}$

After standardization every feature has mean = 0, std = 1, so weight magnitudes are directly comparable.

> **Pipeline rule:** Always fit the scaler on **training data only**, then `transform` both train and test. Fitting on the full dataset leaks test statistics into training.

> **Industry Standard Pattern:** After implementing manual standardization, show the sklearn equivalent:
> ```python
> from sklearn.preprocessing import StandardScaler
> scaler = StandardScaler()
> X_train_scaled = scaler.fit_transform(X_train) # Fit on train only!
> X_test_scaled = scaler.transform(X_test)
> ```
> **When to use:** Always in production. Manual z-score shown for learning only.

In [ ]:
# TODO: Implement this cell
#  (Before vs After comparison)
#
# Steps:
# 1. Call fetch_california_housing() to load the dataset object
# 2. Create a pandas DataFrame: pd.DataFrame(housing.data, columns=housing.feature_names)
# 3. Append target column: df['MedHouseVal'] = housing.target
# 4. Print dataset shape and preview with df.head()
#
# Hint:
#   housing = fetch_california_housing()
#   df = pd.DataFrame(housing.data, columns=housing.feature_names)
#   df['MedHouseVal'] = housing.target

## 0b · Variance Threshold — Dropping Near-Constant Features

A feature that barely changes gives the model nothing to latch onto — it's like trying to predict house prices using a column that says "2.00" for every district. Before proceeding with importance ranking, we check that all features have sufficient variance.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import VarianceThreshold from sklearn.feature_selection
# 2. Compute per-feature variance: X_train_s.var(axis=0)
# 3. Instantiate VarianceThreshold(threshold=0.01) and call .fit(X_train_s)
# 4. Use .get_support() to get the boolean keep-mask
# 5. Visualise variance with a horizontal bar chart
#
# Hint:
#   selector = VarianceThreshold(threshold=0.01)
#   selector.fit(X_train_s)
#   mask = selector.get_support()

## 0c · Understanding Positive Skew — When Features Need Log Transform

**Positive skew** means a distributional asymmetry with a long right tail — most values cluster near the low end, but a few extreme values stretch far to the right. This breaks StandardScaler because the few extreme values inflate the standard deviation (σ), compressing 95% of the data into a narrow band near zero while placing outlier districts at +8σ or +12σ.

**Example:** `Population` ranges from 3 to 35,682. Let's visualize the problem and the log-transform solution.


**Decision Logic Template:**

When you implement feature inspection, include decision logic in your code:

```python
for col in numeric_cols:
 skew = df[col].skew()
 iqr = df[col].quantile(0.75) - df[col].quantile(0.25)
 std = df[col].std()

 # DECISION LOGIC (add this pattern)
 if abs(skew) > 1.0:
 print(f"{col:12s} Skew={skew:5.2f} → Apply log1p + StandardScaler")
 elif iqr / std > 2.5:
 print(f"{col:12s} IQR/std={iqr/std:.2f} → RobustScaler (outlier-resistant)")
 else:
 print(f"{col:12s} Skew={skew:5.2f} → StandardScaler (symmetric)")
```

**Thresholds:**
- |Skew| > 1.0 → log transform needed
- IQR/std > 2.5 → heavy outliers, use RobustScaler
- Otherwise → StandardScaler

In [ ]:
# TODO: Implement this cell
# Hint:
#   from sklearn.preprocessing import PowerTransformer
#   from scipy.stats import skew
#   pop_raw = X_train_raw['Population']
#   pop_log = np.log1p(pop_raw)


## 0d · Weight-Magnitude Comparison — The 28,000× Gap is a Scale Artifact

Let's demonstrate why raw weights can't be compared. We'll fit two models — one on raw features, one on scaled — and show that the weight gap is almost entirely due to input scale, not importance.

In [ ]:
# TODO: Implement this cell
# Hint:
#   model_raw = LinearRegression()
#   model_raw.fit(X_train_raw, y_train)
#   model_scaled = LinearRegression()
#   model_scaled.fit(X_train_s, y_train)


## 1 · Fit the Baseline Model

Data is already loaded and scaled from Section 0. We just fit the Ch.2 LinearRegression on the standardized splits.

In [ ]:
# TODO: Implement this cell
# Hint:
#   X = X_train_raw  # alias — full feature DataFrame for labelling elsewhere
#   model = LinearRegression()
#   model.fit(X_train_s, y_train)
#   y_pred = model.predict(X_test_s)


## 1b · Three-Lens Framework — How to Interpret Feature Rankings

We'll measure feature importance using **three methods**. Each answers a different question, and where their rankings diverge tells the richest diagnostic story:

| Univariate R² | Methods 2+3 (Joint) | Interpretation | Example |
|---|---|---|
| **High** | **High** | Strong, independent, irreplaceable | MedInc — dominates alone *and* in joint model |
| **High** | **Low** | Signal shared with correlated features | AveRooms — standalone power absorbed by AveBedrms |
| **Low** | **High** | Jointly irreplaceable — only works in combination | Lat/Lon — useless alone, critical together for geography |
| **Low** | **Low** | Genuinely uninformative | Population — contributes ~$0 regardless of method |

**Armed with this framework, let's collect all three views:**

## 1c · Filter Methods — Pearson vs Mutual Information

Before fitting any model we can get a directional signal from **filter statistics** — computed directly from the data.

**Pearson ρ** captures linear associations. **Mutual Information** captures *any* statistical dependence: curves, U-shapes, thresholds, clusters.

$$I(X;Y) = \iint p(x,y)\,\log\frac{p(x,y)}{p(x)\,p(y)}\,dx\,dy$$

$p(x,y)$ is the joint density — a 2-D map of where the scatter concentrates. $p(x) \cdot p(y)$ is the independence baseline. The log-ratio measures, at every point, how far the actual density deviates from independence; MI sums those deviations over the whole plane.

**When to use which:**
- **Pearson** — linear or near-linear; score directly comparable to R²
- **MI** — any shape; essential before non-linear models and as a sanity check for linear ones

> **How `sklearn` estimates MI for continuous variables.** `mutual_info_regression` uses the **Kraskov k-NN estimator**: for each sample it measures the distance to its *k*-th nearest neighbour in joint (x, y) space versus in the separate marginal spaces — no binning, no histogram choice. Key settings: `n_neighbors` controls bias-variance (default 3: lower bias but noisier; larger k smooths at the cost of resolution). Always set `random_state` — the estimator adds a tiny perturbation to break ties. MI scores are **not normalised** — use relative magnitude only, never compare across datasets.

> **Industry Standard:** After computing Pearson correlation manually, use:
> ```python
> import pandas as pd
> import seaborn as sns
> corr_matrix = df.corr()
> sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
> ```
> **For Mutual Information:**
> ```python
> from sklearn.feature_selection import mutual_info_regression
> mi_scores = mutual_info_regression(X, y, n_neighbors=3, random_state=42)
> ```

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use default_rng(???) to implement this step
# 2. Use normal(???) to implement this step
# 3. Use sin(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

In [ ]:
# TODO: Implement this cell
# Hint:
#   x_demo = np.linspace(-3, 3, 500)
#   y_parabola  = x_demo ** 2
#   y_linear    = 0.8 * x_demo + np.random.default_rng(42).normal(0, 0.4, 500)
#   y_ushape    = np.sin(x_demo) + np.random.default_rng(42).normal(0, 0.3, 500)


## 2 · Method 1 — Univariate R²

**Question:** If I used only this feature, how much target variance would it explain?

**Shortcut:** For linear regression, univariate R² = ρ(xⱼ, y)². Read it from the correlation matrix — no need to fit 8 models.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use subplots(???) to implement this step
# 2. Use barh(???) to implement this step
# 3. Use text(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

In [ ]:
# TODO: Implement this cell
# Hint:
#   fig, ax = plt.subplots(figsize=(9, 4))
#   colors = ['#1d4ed8' if v > 0.05 else '#94a3b8' for v in univariate_r2.values]
#   bars = ax.barh(univariate_r2.index[::-1], univariate_r2.values[::-1], color=c...
#   for bar, val in zip(bars, univariate_r2.values[::-1]):


## 3 · Method 2 — Standardised Weights (Partial Contribution)

**Question:** Given all other features in the model, how much does each feature contribute?

This is the *partial* effect — what each feature adds above and beyond everything else.

> **Industry Standard:** Standardized weights are computed automatically after StandardScaler:
> ```python
> from sklearn.linear_model import LinearRegression
> scaler = StandardScaler()
> X_scaled = scaler.fit_transform(X_train)
> model = LinearRegression().fit(X_scaled, y_train)
> std_weights = np.abs(model.coef_) # Already standardized!
> ```

In [ ]:
# TODO: Implement this cell
# Hint:
#   std_weights = pd.Series(
#       model.coef_,
#       index=housing.feature_names
#   )


## 4 · The Surprise — Why the Rankings Diverge

Let's directly visualise the ranking gap between the two methods.

In [ ]:
# TODO: Implement this cell
# Hint:
#   uni_norm = univariate_r2 / univariate_r2.max()
#   wt_norm  = abs_weights   / abs_weights.max()
#   comparison = pd.DataFrame({
#       'Univariate R² (alone)':      uni_norm,


## 5 · Feature Correlation Heatmap

Visualising *feature × feature* correlations reveals the collinear pairs **before** computing VIF.

In [ ]:
# TODO: Implement this cell
# Hint:
#   corr_matrix = X_train.corr()
#   fig, ax = plt.subplots(figsize=(8, 6.5))
#   mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # upper triangle ...
#   sns.heatmap(


## 6 · VIF — Quantifying Multicollinearity

VIF measures how much each feature's weight is inflated by its correlation with the **other features**.

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² from regressing feature $j$ on **all other features** (not the target).

> **Industry Standard:** Use statsmodels for VIF calculation:
> ```python
> from statsmodels.stats.outliers_influence import variance_inflation_factor
> vif_scores = [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
> ```
> **When to use:** Part of standard feature engineering pipelines before training.


**Decision Logic Template:**

When you compute VIF, add severity classification:

```python
for feature, vif in vif_data.iterrows():
 # DECISION LOGIC
 if vif > 10:
 verdict = " SEVERE - Drop one of the collinear pair"
 elif vif > 5:
 verdict = " HIGH - Monitor or regularize (Ch.5)"
 elif vif > 3:
 verdict = " MODERATE - Acceptable"
 else:
 verdict = " SAFE"

 print(f"{feature:12s} VIF={vif:5.1f} {verdict}")
```

**VIF Thresholds:**
- VIF > 10 → Severe collinearity, must drop one feature
- VIF 5-10 → High, monitor or use Ridge regularization
- VIF 3-5 → Moderate, acceptable
- VIF < 3 → Safe, independent signal

In [ ]:
# TODO: Implement this cell
# Hint:
#   vif_df = pd.DataFrame({
#       'Feature': housing.feature_names,
#       'VIF': [variance_inflation_factor(X_train.values, i)
#               for i in range(X_train.shape[1])]


In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use Series(???) to implement this step
# 2. Use permutation_importance(???) to implement this step
# 3. Use sort_values(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

## 7 · Method 3 — Permutation Importance

**Question:** If I *scramble* this feature on the test set (destroying its signal), how much does MAE rise?

This is the **most reliable** and model-agnostic method. Crucially, the model is never retrained — you're measuring how badly the model's existing weights are handicapped when a feature's signal is destroyed. This makes it a pure test of the model's *reliance* on each feature, not just correlation or fitted weights.

> **Industry Standard:** Use sklearn's permutation_importance:
> ```python
> from sklearn.inspection import permutation_importance
> result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)
> perm_importance = result.importances_mean
> ```
> **When to use:** Post-training model interpretation. Standard in AutoML libraries.


**Decision Logic Template:**

After computing permutation importance, add drop candidate logic:

```python
for feature, perm_imp in importance_data.iterrows():
 # DECISION LOGIC
 if perm_imp < 0.005: # Near-zero threshold
 verdict = " DROP CANDIDATE - No measurable signal"
 elif perm_imp < 0.05:
 verdict = " WEAK - Verify with VIF before dropping"
 else:
 verdict = " KEEP - Meaningful contribution"

 print(f"{feature:12s} Δ MAE={perm_imp*1000:4.1f}k {verdict}")
```

**Permutation Thresholds:**
- < 0.005 (< $5 MAE rise) → Drop candidate
- 0.005-0.05 → Weak but may be important jointly
- > 0.05 → Keep, meaningful signal

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use add_subplot(???) to implement this step
# 2. Use scatter(???) to implement this step
# 3. Use set_ylabel(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

In [ ]:
# TODO: Implement this cell
# Hint:
#   fig, ax = plt.subplots(figsize=(9, 4))
#   colors = ['#1d4ed8' if v > 0.01 else '#94a3b8' for v in perm_imp.values]
#   ax.barh(
#       perm_imp.index[::-1], perm_imp.values[::-1],


## 7b · Permutation Shuffle Loop — Visualizing How It Works

Let's demonstrate the shuffle loop on a small sample to show exactly what permutation importance measures. We'll take 10 test samples, shuffle one feature (MedInc), and show that predictions change even though the model weights stay frozen.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use n(???) to implement this step
# 2. Use figure(???) to implement this step
# 3. Use GridSpec(???) to implement this step
#
# Hint:
#   # see solution cell for the required API calls

## 8 · Three-View Dashboard

Side-by-side comparison of all three methods. Features where rankings diverge tell the richest story.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Call train_test_split(X, y, test_size=0.2, random_state=SEED)
# 2. Instantiate StandardScaler()
# 3. scaler.fit_transform(X_train) to get scaled train features
# 4. scaler.transform(X_test) — no fit here to avoid data leakage
#
# Hint:
#   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=???, random_state=???)
#   scaler = StandardScaler()
#   X_train_s = scaler.fit_transform(X_train)  # fit+transform on train only
#   X_test_s  = scaler.transform(X_test)       # transform only — no leakage

In [ ]:
# TODO: Implement this cell
# Hint:
#   fig = plt.figure(figsize=(15, 4.5))
#   gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.45)
#   methods = [
#       ('Univariate R²\n(alone)', univariate_r2, '#94a3b8'),


## 9 · Multicollinearity Deep Dive — AveRooms vs AveBedrms

Visualise *why* the two room features cause weight instability.

In [ ]:
# TODO: Implement this cell
# Hint:
#   fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#   axes[0].scatter(
#       X_train['AveRooms'].clip(0, 10),
#       X_train['AveBedrms'].clip(0, 5),


## 9b · Joint Feature Importance — Cooperation vs Competition

VIF catches the *competition* case: two features measure the same thing and their weights blow up. The opposite exists too: two features can be **individually weak but jointly irreplaceable** — the *cooperation* case.

**Diagnostic — joint permutation importance**: shuffle both features simultaneously and compare the drop to the sum of their individual drops.

$$\Delta_{\text{interact}}(j,k) = \pi_{jk} - \pi_j - \pi_k$$

- $\Delta > 0$ → the model is using their *interaction*; the features cooperate (Lat/Long case)
- $\Delta \approx 0$ → additive independence
- $\Delta < 0$ → the features are substitutes; one is largely redundant (AveRooms/AveBedrms case)

In [ ]:
# TODO: Implement this cell
# Hint:
#   def joint_perm_importance(model, X, y, feat_a_idx, feat_b_idx, n_repeats=20, ...
#       """Shuffle feat_a and feat_b simultaneously and return mean Δ MAE."""
#       rng = np.random.default_rng(random_state)
#       baseline = mean_absolute_error(y, model.predict(X))


## 10 · Action Items for Ch.4 & Ch.5

Based on the three-view dashboard, what do we know about pursuing the $40k MAE target?

In [ ]:
# TODO: Implement this cell
# Hint:
#   print('═' * 65)
#   print(' FEATURE IMPORTANCE AUDIT — ACTION ITEMS')
#   print('═' * 65)
#   print('''


## Summary

| Method | Top feature | Bottom feature | Best use |
|---|---|---|---|
| Univariate R² | MedInc (0.47) | Population (0.001) | First-pass scan; quick ranking |
| Std \|weight\| | Latitude (0.89) | Population (0.01) | Final model inspection; requires standardisation |
| Permutation | MedInc (+$18k) | Population (+$0.1k) | Most reliable; model-agnostic; use on test set |

**Key takeaways:**
1. **MedInc** is the strongest individual predictor regardless of method.
2. **Latitude and Longitude** are jointly irreplaceable — low alone, high together.
3. **AveRooms / AveBedrms** are collinear (VIF ≈ 7) — their weight split is arbitrary.
4. **Population** contributes virtually nothing and is a candidate for removal.
5. Always inspect all three views; a feature cheap in one can be critical in another.

**Next:** Ch.4 — Polynomial Features: add `MedInc²`, `MedInc × Latitude`, `MedInc × Longitude` to push MAE toward $48k.